In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import random
%pip install pybaseball

from pybaseball import team_batting, team_pitching, standings


  Obtaining dependency information for pybaseball from https://files.pythonhosted.org/packages/bb/66/5ef47f5830570a30afbdfbd741cdf3e5a1a31c4c588514ab69bc074e8704/pybaseball-2.2.7-py3-none-any.whl.metadata
  Obtaining dependency information for pygithub>=1.51 from https://files.pythonhosted.org/packages/07/ba/7049ce39f653f6140aac4beb53a5aaf08b4407b6a3019aae394c1c5244ff/pygithub-2.8.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/426.1 kB ? eta -:--:--
   ---- ----------------------------------- 51.2/426.1 kB 1.3 MB/s eta 0:00:01
   ------------------------------------ --- 389.1/426.1 kB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 426.1/426.1 kB 5.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/432.7 kB ? eta -:--:--
   --------------------------------------- 432.7/432.7 kB 28.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


INITIAL CREATION


SALARY CAP AND FLOOR GIVEN BY NHL FORMULA,
MORE INFO ON [REDDIT LINK](https://www.reddit.com/r/baseball/comments/1iy5eor/what_mlbs_hypothetical_salary_floor_and_cap_would/)

In [ ]:
SALARY_CAP = 231800000
SALARY_FLOOR = 171400000

In [ ]:
###  DATABASE Creation ###
# Fetch Offense (Batting)
# We only need the Team name and 'R' (Runs Scored)
batting = team_batting(2025)
batting = batting[['Team', 'R']].rename(columns={'R': 'RS'})


# Fetch Defense (Pitching)
pitching = team_pitching(2025)
pitching = pitching[['Team', 'W', 'L', 'R']].rename(columns={'R': 'RA'})


# Merge to create standings dataframe
df_season = pd.merge(batting, pitching, on='Team')

# Calculate Win Percentage for reference
df_season['Win_Pct'] = df_season['W'] / (df_season['W'] + df_season['L'])

### Salaries DataFrame Creation ###
# payroll data estimated from fangraphs 2026 luxury tax projections
data_2026_payrolls = {
    'Team': [
        'LAD', 'NYM', 'NYY', 'PHI', 'TOR', 'SDP', 'BOS', 'HOU', 'TEX', 'ATL',
        'CHC', 'SFG', 'LAA', 'ARI', 'SEA', 'DET', 'KCR', 'BAL', 'STL', 'COL',
        'CIN', 'MIL', 'MIN', 'WSN', 'CLE', 'ATH', 'PIT', 'TBR', 'CHW', 'MIA'
    ],
    'Payroll': [
        404000000, 366000000, 333000000, 317000000, 316000000, 264000000, 264000000, 233000000, 206000000, 260000000,
        244000000, 232000000, 195000000, 223000000, 186000000, 246000000, 185000000, 207000000, 113000000, 137000000,
        151000000, 135000000, 129000000, 116000000, 98000000, 146000000, 127000000, 109000000, 108000000, 84000000
    ]
}

df_salaries_2026 = pd.DataFrame(data_2026_payrolls)


# Merge payroll data directly on 'Team' and select relevant columns
master_df = pd.merge(df_season, df_salaries_2026, left_on='Team', right_on='Team', how='left')
master_df = master_df[['Team', 'W', 'L', 'RS', 'RA', 'Payroll', 'Win_Pct']]

display(master_df)


In [ ]:
### Linear Regression ###

x_money = master_df['Payroll'] / 1000000
y_rs = master_df['RS']
y_ra = master_df['RA']

# slopes for regression
slope_rs, intercept_rs, _, _, _ = stats.linregress(x_money, y_rs)
slope_ra, intercept_ra, _, _, _ = stats.linregress(x_money, y_ra)

# prediction and residuals
master_df['Pred_RS'] = (x_money * slope_rs) + intercept_rs
master_df['Resid_RS'] = master_df['RS'] - master_df['Pred_RS']

master_df['Pred_RA'] = (x_money * slope_ra) + intercept_ra
master_df['Resid_RA'] = master_df['RA'] - master_df['Pred_RA']


print(f"OFFENSE: $10 Million buys {slope_rs * 10:.2f} extra Runs Scored")
print(f"DEFENSE: $10 Million prevents {slope_ra * 10 * -1:.2f} Runs Allowed")
print("-" * 50)

SIMULATIONS

In [ ]:
### CAP ONLY MONTE CARLO iter1###


# --- CONFIGURATION FOR 2027 PROPOSAL ---
SALARY_CAP = SALARY_CAP  # Potential 2027 Cap
SIMULATIONS = 1000       # Number of seasons to run

# 1. PREPARE THE DATA
sim_df = master_df.copy()

# 2. CAP LOGIC: The "Hard Cap" Cut
# We track how much money is removed from the ecosystem
total_excess_money = 0

def apply_hard_cap(payroll):
    global total_excess_money
    if payroll > SALARY_CAP:
        excess = payroll - SALARY_CAP
        total_excess_money += excess
        return SALARY_CAP
    return payroll

sim_df['Sim_Payroll'] = sim_df['Payroll'].apply(apply_hard_cap)

# 3. REDISTRIBUTION LOGIC
# The proposal usually implies spreading the wealth.
# We distribute the collected "Luxury Tax" evenly to the bottom 15 teams.
print(f"--- 2027 PROPOSAL SIMULATION ---")
print(f"Hard Cap Limit: ${SALARY_CAP:,.0f}")
print(f"Total Money Removed/Redistributed: ${total_excess_money:,.0f}")

bottom_15_teams = sim_df.nsmallest(15, 'Payroll').index
share_per_team = total_excess_money / 15
sim_df.loc[bottom_15_teams, 'Sim_Payroll'] += share_per_team

print(f"Benefit per Small Market Team: +${share_per_team:,.0f}")

# 4. CALCULATE NEW PERFORMANCE (The Regression Application)
# New Runs = (New Money * Slope) + Intercept + Old Residuals
sim_df['New_RS'] = ((sim_df['Sim_Payroll'] / 1000000) * slope_rs) + intercept_rs + sim_df['Resid_RS']
sim_df['New_RA'] = ((sim_df['Sim_Payroll'] / 1000000) * slope_ra) + intercept_ra + sim_df['Resid_RA']

# 5. PYTHAGOREAN WIN PROBABILITY
sim_df['New_Win_Pct'] = (sim_df['New_RS']**1.83) / ((sim_df['New_RS']**1.83) + (sim_df['New_RA']**1.83))

# 6. MONTE CARLO SIMULATION
results = {team: [] for team in sim_df['Team']}

for i in range(SIMULATIONS):
    for idx, row in sim_df.iterrows():
        # Simulate 162 games
        wins = np.random.binomial(n=162, p=row['New_Win_Pct'])
        results[row['Team']].append(wins)

# 7. VISUALIZATION
SALARY_CAP_DF = pd.DataFrame({
    'Team': results.keys(),
    'Real_Wins': sim_df['W'],
    'Sim_Wins': [np.mean(val) for val in results.values()],
    'Payroll_Cut': (sim_df['Payroll'] - sim_df['Sim_Payroll']) / 1000000
})

SALARY_CAP_DF['Win_Diff'] = SALARY_CAP_DF['Sim_Wins'] - SALARY_CAP_DF['Real_Wins']
SALARY_CAP_DF = SALARY_CAP_DF.sort_values('Win_Diff')



In [ ]:
# print the plot
plt.figure(figsize=(14, 8))
colors = ['#d62728' if x < 0 else '#2ca02c' for x in SALARY_CAP_DF['Win_Diff']]
sns.barplot(
    x='Win_Diff',
    y='Team',
    hue='Team',       # <--- New requirement
    data=SALARY_CAP_DF,
    palette=colors,
    legend=False      # <--- New requirement
)
plt.title(f"Impact of Proposed ${SALARY_CAP/1000000}M Hard Cap (2027 Season)", fontsize=16)
plt.xlabel("Change in Wins", fontsize=12)
plt.axvline(0, color='black', linewidth=1)
plt.show()

# New plot: Distribution of simulated wins for the CAP ONLY scenario

# 1. Convert Raw Results to DataFrame (Rows=Simulations, Cols=Teams)
raw_sim_cap_df = pd.DataFrame(results)

# 2. Calculate order for sorting (by Mean Wins)
sorted_order = raw_sim_cap_df.mean().sort_values(ascending=False).index

# 3. Visualization: Box Plot (Distribution of Outcomes)
plt.figure(figsize=(16, 8))
sns.boxplot(data=raw_sim_cap_df, order=sorted_order, color="lightskyblue")
plt.title(f"Distribution of Simulated Season Outcomes for CAP ONLY ({SIMULATIONS} Iterations)\nBox = 25th-75th %ile | Whiskers = Range", fontsize=16)
plt.ylabel("Simulated Wins", fontsize=12)
plt.xlabel("Team", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()

In [ ]:
### FLOOR ONLY MONTE CARLO iter1###

# --- CONFIGURATION: SALARY FLOOR ONLY ---
SALARY_FLOOR = SALARY_FLOOR  # Minimum required spend
SIMULATIONS = 1000        # Seasons to run

# 1. PREPARE THE DATA
# Ensure we start with a clean 30-team dataframe
sim_df = master_df.sort_values('Payroll', ascending=False).drop_duplicates('Team').copy()

# 2. FLOOR LOGIC: The "Forced Spending" Rule
# If a team spends less than the floor, they must increase payroll to meet it.
# (We assume this money comes from the owners, not a tax on rich teams).

def apply_floor(payroll):
    if payroll < SALARY_FLOOR:
        return SALARY_FLOOR
    return payroll

sim_df['Sim_Payroll'] = sim_df['Payroll'].apply(apply_floor)

# Calculate how much money was forced into the league
total_investment = (sim_df['Sim_Payroll'] - sim_df['Payroll']).sum()
print(f"--- SALARY FLOOR SIMULATION (${SALARY_FLOOR/1000000:.0f}M) ---")
print(f"Total New Money Invested by Owners: ${total_investment:,.0f}")

# 3. CALCULATE NEW PERFORMANCE
# New Runs = (New Money * Slope) + Intercept + Old Residuals
sim_df['New_RS'] = ((sim_df['Sim_Payroll'] / 1000000) * slope_rs) + intercept_rs + sim_df['Resid_RS']
sim_df['New_RA'] = ((sim_df['Sim_Payroll'] / 1000000) * slope_ra) + intercept_ra + sim_df['Resid_RA']

# 4. PYTHAGOREAN WIN PROBABILITY
sim_df['New_Win_Pct'] = (sim_df['New_RS']**1.83) / ((sim_df['New_RS']**1.83) + (sim_df['New_RA']**1.83))

# 5. MONTE CARLO SIMULATION
results = {team: [] for team in sim_df['Team']}

print(f"Simulating {SIMULATIONS} seasons...")
for i in range(SIMULATIONS):
    for idx, row in sim_df.iterrows():
        wins = np.random.binomial(n=162, p=row['New_Win_Pct'])
        results[row['Team']].append(wins)

# 6. VISUALIZATION
# Build the results table safely
results_df = pd.DataFrame({
    'Team': list(results.keys()),
    'Sim_Wins': [np.mean(val) for val in results.values()]
})

SALARY_FLOOR_DF = pd.merge(results_df, sim_df[['Team', 'W', 'Payroll', 'Sim_Payroll']], on='Team')
SALARY_FLOOR_DF['Real_Wins'] = SALARY_FLOOR_DF['W']
SALARY_FLOOR_DF['Win_Diff'] = SALARY_FLOOR_DF['Sim_Wins'] - SALARY_FLOOR_DF['Real_Wins']
SALARY_FLOOR_DF['Payroll_Increase'] = (SALARY_FLOOR_DF['Sim_Payroll'] - SALARY_FLOOR_DF['Payroll']) / 1000000

SALARY_FLOOR_DF = SALARY_FLOOR_DF.sort_values('Win_Diff')

In [ ]:
### Plot Floor ###
plt.figure(figsize=(14, 8))
colors = ['#d62728' if x < 0 else '#2ca02c' for x in SALARY_FLOOR_DF['Win_Diff']]

sns.barplot(
    x='Win_Diff',
    y='Team',
    hue='Team',
    data=SALARY_FLOOR_DF,
    palette=colors,
    legend=False
)

plt.title(f"Impact of ${SALARY_FLOOR/1000000}M Salary Floor Only", fontsize=16)
plt.xlabel("Change in Wins", fontsize=12)
plt.axvline(0, color='black', linewidth=1)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()
# 1. Convert Raw Results to DataFrame (Rows=Simulations, Cols=Teams)
raw_sim_cap_df = pd.DataFrame(results)

# 2. Calculate order for sorting (by Mean Wins)
sorted_order = raw_sim_cap_df.mean().sort_values(ascending=False).index

# 3. Visualization: Box Plot (Distribution of Outcomes)
plt.figure(figsize=(16, 8))
sns.boxplot(data=raw_sim_cap_df, order=sorted_order, color="lightskyblue")
plt.title(f"Distribution of Simulated Season Outcomes for CAP ONLY ({SIMULATIONS} Iterations)\nBox = 25th-75th %ile | Whiskers = Range", fontsize=16)
plt.ylabel("Simulated Wins", fontsize=12)
plt.xlabel("Team", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()


In [ ]:
### SALARY FLOOR W/ OWNERSHIP HABITS ###

# --- CONFIGURATION ---
SALARY_FLOOR = SALARY_FLOOR
SIMULATIONS = 1000

# 1. DEFINE OWNER HABITS (The Volatility Index)
def get_spending_volatility(current_payroll):

    # Returns the standard deviation (swing) of an owner's spending.
    # - Rich teams (>$250M) swing wildly (+/- $30M).
    # - Mid teams ($150M-$250M) swing moderately (+/- $15M).
    # - Poor teams (<$150M) have strict budgets (+/- $8M).

    if current_payroll > 250000000:
        return 30000000
    elif current_payroll > 150000000:
        return 15000000
    else:
        return 8000000

# Prepare Data
sim_df = master_df.sort_values('Payroll', ascending=False).drop_duplicates('Team').copy()
sim_df['Volatility'] = sim_df['Payroll'].apply(get_spending_volatility)

# 2. THE STOCHASTIC SIMULATION LOOP
print(f"Simulating {SIMULATIONS} seasons with Variable Owner Spending...")

# We need to track Wins AND the actual Payroll used (since it changes every season)
results_wins = {team: [] for team in sim_df['Team']}
results_payrolls = {team: [] for team in sim_df['Team']}

for i in range(SIMULATIONS):
    # A. ROLL THE DICE ON PAYROLL
    # Create a random spending number for this specific season
    # Logic: Start at current payroll, vary by volatility
    raw_payrolls = np.random.normal(sim_df['Payroll'], sim_df['Volatility'])

    # B. APPLY THE FLOOR
    # If the random roll is below the floor, the league forces them up.
    # If the random roll is ABOVE the floor (e.g., Royals decide to splurge), they keep it.
    final_payrolls = np.maximum(raw_payrolls, SALARY_FLOOR)

    # C. CALCULATE TEAM STRENGTH (Vectorized)
    # Note: We must recalculate runs every time because payroll changed
    current_runs = ((final_payrolls / 1000000) * slope_rs) + intercept_rs + sim_df['Resid_RS']
    current_allowed = ((final_payrolls / 1000000) * slope_ra) + intercept_ra + sim_df['Resid_RA']

    # D. PLAY THE SEASON
    win_pcts = (current_runs**1.83) / ((current_runs**1.83) + (current_allowed**1.83))
    season_wins = np.random.binomial(162, win_pcts)

    # E. STORE DATA
    for team, wins, pay in zip(sim_df['Team'], season_wins, final_payrolls):
        results_wins[team].append(wins)
        results_payrolls[team].append(pay)

# 3. ANALYSIS & VISUALIZATION
# Average out the 1,000 different seasons
HABITS_FLOOR_DF = pd.DataFrame({
    'Team': results_wins.keys(),
    'Sim_Wins': [np.mean(val) for val in results_wins.values()],
    'Sim_Avg_Payroll': [np.mean(val) for val in results_payrolls.values()]
})

# Merge with original data
HABITS_FLOOR_DF = pd.merge(HABITS_FLOOR_DF, sim_df[['Team', 'W', 'Payroll']], on='Team')
HABITS_FLOOR_DF['Win_Diff'] = HABITS_FLOOR_DF['Sim_Wins'] - HABITS_FLOOR_DF['W']
HABITS_FLOOR_DF['Payroll_Diff'] = (HABITS_FLOOR_DF['Sim_Avg_Payroll'] - HABITS_FLOOR_DF['Payroll']) / 1000000
HABITS_FLOOR_DF = HABITS_FLOOR_DF.sort_values('Win_Diff')

In [ ]:
### PLOT 1: Scatter (Owner Habits) ###
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=HABITS_FLOOR_DF,
    x='Payroll_Diff',
    y='Win_Diff',
    hue='Team',
    s=100,
    legend=False
)

# Add reference lines
plt.axhline(0, color='black', linewidth=1)
plt.axvline(0, color='black', linewidth=1)
plt.title("The 'Owner Habits' Model: Spending Increases vs. Win Increases", fontsize=16)
plt.xlabel("Average Increase in Payroll ($ Millions)", fontsize=12)
plt.ylabel("Increase in Wins", fontsize=12)
plt.grid(True, alpha=.3)

# Label the most interesting teams
for i in range(HABITS_FLOOR_DF.shape[0]):
    row = HABITS_FLOOR_DF.iloc[i]
    if abs(row['Win_Diff']) > 5 or row['Payroll_Diff'] > 20:
        plt.text(
            row['Payroll_Diff']+1,
            row['Win_Diff'],
            row['Team'],
            fontsize=9,
            weight='bold'
        )

plt.show()

### PLOT 2: Bar (Win Differential) ###
plt.figure(figsize=(14, 8))
colors = ['#d62728' if x < 0 else '#2ca02c' for x in HABITS_FLOOR_DF['Win_Diff']]

sns.barplot(
    x='Win_Diff',
    y='Team',
    hue='Team',
    data=HABITS_FLOOR_DF,
    palette=colors,
    legend=False
)

plt.title(f"Impact of Salary Floor ($142.5M) with Owner Habits", fontsize=16)
plt.xlabel("Change in Wins", fontsize=12)
plt.axvline(0, color='black', linewidth=1)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

HABITS_FLOOR_DF.head()

In [ ]:
year_wins = team_pitching(2021)
wins = year_wins[["Team","W"]]
wins

In [ ]:
# --- CONFIGURATION ---
SALARY_FLOOR = SALARY_FLOOR
SIMULATIONS = 1000

# 1. BUILD THE "5-YEAR PERFORMANCE" DATABASE
print("Fetching historical standings (2021-2024)...")

# Fetch 2021-2024
history_wins = {}
for year in range(2021, 2025):
    try:
        year_wins = team_pitching(year)
        wins = year_wins[["Team","W"]]
        for index, row in wins.iterrows():
          team_abbr = row['Team']


          if team_abbr == 'OAK':
              team_abbr = 'ATH'
          history_wins[team_abbr] = history_wins.get(team_abbr, 0) + row['W']

    except Exception as e:
        print(f"Warning: Could not fetch {year} data. Using averages.")

# Add your 2025 Data (from master_df)
for index, row in master_df.iterrows():
    team = row['Team']
    # We add 2025 wins to the tally
    history_wins[team] = history_wins.get(team, 0) + row['W']

# 2. DEFINE THE "Prestige SCORE"

  # A. 2025 PRE-SEASON ODDS (The Expectation)
playoff_odds_2025 = {
    'LAD': 0.98, 'ATL': 0.92, 'HOU': 0.85, 'NYY': 0.82, 'PHI': 0.78,
    'BAL': 0.65, 'TEX': 0.60, 'TOR': 0.58, 'SEA': 0.52, 'MIN': 0.50,
    'TBR': 0.45, 'STL': 0.42, 'CHC': 0.40, 'SFG': 0.38, 'SDP': 0.35,
    'ARI': 0.32, 'NYM': 0.30, 'CLE': 0.28, 'MIL': 0.25, 'CIN': 0.20,
    'BOS': 0.18, 'DET': 0.12, 'MIA': 0.08, 'KCR': 0.08, 'LAA': 0.05,
    'PIT': 0.04, 'WSN': 0.02, 'OAK': 0.01, 'COL': 0.00, 'CHW': 0.00
}

# B. REALITY: WHO MADE IT IN 2025?
made_playoffs_2025 = [
    'LAD', 'PHI', 'MIL', 'SDP', 'CIN', 'BOS', # NL
    'NYY', 'TOR', 'SEA', 'CHC', 'CLE', 'DET'  # AL
]

# C. DYNASTY HISTORY (2021-2024 Winners)
WS_CONTENDERS = ['LAD', 'NYY', 'TEX', 'ARI', 'HOU', 'PHI', 'ATL', 'TOR']



# --- 2. THE COMBINED EFFICIENCY LOGIC ---

def desirability(team_abbr, total_5yr_wins, payroll_rank):

    odds = playoff_odds_2025.get(team_abbr, 0.01)
    avg_wins = 405
    win_diff = total_5yr_wins - avg_wins

    # --- PART A: BASE EFFICIENCY (Expectation Logic) ---
    base_efficiency = 1.0

    if team_abbr in made_playoffs_2025:
        if odds<.5:
            base_efficiency = 1 + (1 - odds)*.2
        else:
            base_efficiency = 1 + odds*.2

    else:
        if odds<.7:
            base_efficiency = 1 + odds*.2
        else:
            base_efficiency = 1 - odds*.2

    # --- PART B: DYNASTY BONUSES (The "Rest" of the Calculation) ---
    bonus = 0.0

    # 1. Winning Culture (5-Year Wins)
    # Every 10 wins above/below avg (405) adds/subtracts 1%
    bonus += (win_diff) / 1000

    # 2. Championship Glow (Recent WS)
    if team_abbr in WS_CONTENDERS:
        bonus += 0.05 # +5%

    # 3. Payroll Bonus (Top 10 / Bottom 10)
    if payroll_rank <= 10:
        bonus += 0.05
    elif payroll_rank > 20:
        bonus -= 0.05


    # --- PART C: COMBINE ---
    return base_efficiency + bonus

# Apply to DataFrame
desire_df = master_df.sort_values('Payroll', ascending=False).drop_duplicates('Team').copy()
desire_df['5Yr_Wins'] = desire_df['Team'].map(history_wins).fillna(405) # Fill missing with avg
desire_df['Volatility'] = desire_df['Payroll'].apply(get_spending_volatility)

# Calculate Payroll Rank
desire_df['Payroll_Rank'] = desire_df['Payroll'].rank(ascending=False, method='min')

# Calculate Efficiency
desire_df['Efficiency_Factor'] = desire_df.apply(lambda x: desirability(x['Team'], x['5Yr_Wins'], x['Payroll_Rank']), axis=1)

print("\n--- TOP 5 MOST DESIRABLE DESTINATIONS ---")
display(desire_df[['Team', '5Yr_Wins', 'Efficiency_Factor']].sort_values('Efficiency_Factor', ascending=False).head(5))

print("\n--- LEAST DESIRABLE DESTINATIONS ---")
display(desire_df[['Team', '5Yr_Wins', 'Efficiency_Factor']].sort_values('Efficiency_Factor', ascending=True).head(5))


# 3. RUN THE SIMULATION
results_wins_desire = {team: [] for team in desire_df['Team']}

print(f"\nSimulating {SIMULATIONS} seasons with Dynasty Factors...")

for i in range(SIMULATIONS):
    # A. Volatility Spending
    raw_payrolls = np.random.normal(desire_df['Payroll'], desire_df['Volatility'])
    final_payrolls = np.maximum(raw_payrolls, SALARY_FLOOR)

    # B. Apply Dynasty Efficiency
    # Money buys more WAR for desirable teams
    eff_payrolls = final_payrolls * desire_df['Efficiency_Factor']

    # C. Calculate Performance
    current_runs = ((eff_payrolls / 1000000) * slope_rs) + intercept_rs + desire_df['Resid_RS']
    current_allowed = ((eff_payrolls / 1000000) * slope_ra) + intercept_ra + desire_df['Resid_RA']

    # D. Play Games
    win_pcts = (current_runs**1.83) / ((current_runs**1.83) + (current_allowed**1.83))
    season_wins = np.random.binomial(162, win_pcts)

    for team, wins in zip(desire_df['Team'], season_wins):
        results_wins_desire[team].append(wins)

# 4. VISUALIZE
DESIRE_DF = pd.DataFrame({
    'Team': results_wins_desire.keys(),
    'Desire_Wins': [np.mean(val) for val in results_wins_desire.values()]
})

DESIRE_DF = pd.merge(DESIRE_DF, desire_df[['Team', 'W', 'Efficiency_Factor']], on='Team')
DESIRE_DF['Win_Diff'] = DESIRE_DF['Desire_Wins'] - DESIRE_DF['W']
DESIRE_DF = DESIRE_DF.sort_values('Win_Diff', ascending=False)

In [ ]:
plt.figure(figsize=(14, 8))
colors = ['#d62728' if x < 0 else '#2ca02c' for x in DESIRE_DF['Win_Diff']]

# Create a custom label for the bars
plot_data = DESIRE_DF.copy()
plot_data['Label'] = plot_data['Team'] + " (" + plot_data['Efficiency_Factor'].round(2).astype(str) + "x)"

sns.barplot(x='Win_Diff', y='Label', hue='Team', data=plot_data, palette=colors, legend=False)

plt.title(f"Impact of 'Dynasty Factors' (Past 5 Years Performance)", fontsize=16)
plt.xlabel("Change in Wins", fontsize=12)
plt.ylabel("Team (Efficiency Multiplier)", fontsize=12)
plt.axvline(0, color='black', linewidth=1)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()


# 1. Convert Raw Results to DataFrame (Rows=Simulations, Cols=Teams)
raw_sim_cap_df = pd.DataFrame(results)

# 2. Calculate order for sorting (by Mean Wins)
sorted_order = raw_sim_cap_df.mean().sort_values(ascending=False).index

# 3. Visualization: Box Plot (Distribution of Outcomes)
plt.figure(figsize=(16, 8))
sns.boxplot(data=raw_sim_cap_df, order=sorted_order, color="lightskyblue")
plt.title(f"Distribution of Simulated Season Outcomes for Efficiency ({SIMULATIONS} Iterations)\nBox = 25th-75th %ile | Whiskers = Range", fontsize=16)
plt.ylabel("Simulated Wins", fontsize=12)
plt.xlabel("Team", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()



In [ ]:
### Historical Data Cleaning
# 1. Load Data
df_history = pd.read_csv('/content/Year,Rank,Team,Payroll.csv')

# 2. Clean Payroll Column
def clean_payroll(val):
    if isinstance(val, str):
        val = val.replace('$', '').replace(',', '')
        if 'M' in val:
            return float(val.replace('M', '')) * 1_000_000
    return float(val)

if df_history['Payroll'].dtype == object:
    df_history['Payroll'] = df_history['Payroll'].apply(clean_payroll)

# 3. Standardize Team Names
team_map = {
    'Angels': 'LAA', 'Arizona Diamondbacks': 'ARI', 'Astros': 'HOU', 'Athletics': 'ATH', 'Atlanta Braves': 'ATL',
    'Baltimore Orioles': 'BAL', 'Blue Jays': 'TOR', 'Boston Red Sox': 'BOS', 'Braves': 'ATL', 'Brewers': 'MIL',
    'Cardinals': 'STL', 'Chicago Cubs': 'CHC', 'Chicago White Sox': 'CHW', 'Cincinnati Reds': 'CIN', 'Cleveland Guardians': 'CLE',
    'Cleveland Indians': 'CLE', 'Colorado Rockies': 'COL', 'Cubs': 'CHC', 'Detroit Tigers': 'DET', 'Diamondbacks': 'ARI',
    'Dodgers': 'LAD', 'Giants': 'SFG', 'Guardians': 'CLE', 'Houston Astros': 'HOU', 'Kansas City Royals': 'KCR',
    'Los Angeles Angels': 'LAA', 'Los Angeles Dodgers': 'LAD', 'Mariners': 'SEA', 'Marlins': 'MIA', 'Mets': 'NYM',
    'Miami Marlins': 'MIA', 'Milwaukee Brewers': 'MIL', 'Minnesota Twins': 'MIN', 'Nationals': 'WSN', 'New York Mets': 'NYM',
    'New York Yankees': 'NYY', 'Oakland Athletics': 'ATH', 'Orioles': 'BAL', 'Padres': 'SDP', 'Philadelphia Phillies': 'PHI',
    'Phillies': 'PHI', 'Pirates': 'PIT', 'Pittsburgh Pirates': 'PIT', 'Rangers': 'TEX', 'Rays': 'TBR',
    'Red Sox': 'BOS', 'Reds': 'CIN', 'Rockies': 'COL', 'Royals': 'KCR', 'San Diego Padres': 'SDP',
    'San Francisco Giants': 'SFG', 'Seattle Mariners': 'SEA', 'St. Louis Cardinals': 'STL', 'Tampa Bay Rays': 'TBR', 'Texas Rangers': 'TEX',
    'Tigers': 'DET', 'Toronto Blue Jays': 'TOR', 'Twins': 'MIN', 'Washington Nationals': 'WSN', 'White Sox': 'CHW',
    'Yankees': 'NYY'
}
df_history['Team_Abbr'] = df_history['Team'].map(team_map)

# 4. Calculate Historical Volatility
volatility_df = df_history.groupby('Team_Abbr')['Payroll'].std().reset_index()
volatility_df.rename(columns={'Payroll': 'Hist_Volatility'}, inplace=True)

# 5. Handle Missing Data
mean_volatility = volatility_df['Hist_Volatility'].mean()
volatility_df['Hist_Volatility'] = volatility_df['Hist_Volatility'].fillna(mean_volatility)

print("Data Cleaning Complete. Volatility Calculated:")
display(volatility_df.head())

In [ ]:
# --- 1. UPDATE MASTER DATASET ---

# Define columns to be added/updated
vol_cols = ['Team_Abbr', 'Hist_Volatility']
eff_cols = ['Efficiency_Factor']

# CLEANUP A: Remove Volatility columns if they exist (to prevent duplicates/_x suffixes)
cols_to_drop_vol = [c for c in vol_cols if c in master_df.columns]
if cols_to_drop_vol:
    master_df = master_df.drop(columns=cols_to_drop_vol)

# A. Merge Historical Volatility (from volatility_df)
# volatility_df has ['Team_Abbr', 'Hist_Volatility']
master_df = pd.merge(master_df, volatility_df[['Team_Abbr', 'Hist_Volatility']],
                    left_on='Team', right_on='Team_Abbr', how='left')

# Fill missing volatility with mean
master_df['Hist_Volatility'] = master_df['Hist_Volatility'].fillna(master_df['Hist_Volatility'].mean())

# CLEANUP B: Remove Efficiency column if it exists
cols_to_drop_eff = [c for c in eff_cols if c in master_df.columns]
if cols_to_drop_eff:
    master_df = master_df.drop(columns=cols_to_drop_eff)

# B. Merge Dynasty Efficiency (from desire_df)
# desire_df has ['Team', 'Efficiency_Factor']
master_df = pd.merge(master_df, desire_df[['Team', 'Efficiency_Factor']], on='Team', how='left')
master_df['Efficiency_Factor'] = master_df['Efficiency_Factor'].fillna(1.0)

print("--- MASTER DATASET UPDATED ---")
display(master_df[['Team', 'Payroll', 'Hist_Volatility', 'Efficiency_Factor']].head())

In [ ]:
### HISTORICAL VOLATILITY MONTE CARLO ###

# Configuration
SALARY_FLOOR = SALARY_FLOOR
SIMULATIONS = 1000

print(f"--- Running Simulation with Historical Volatility ---")
print(f"Salary Floor: ${SALARY_FLOOR:,.0f}")

# 1. Prepare Local Data for Simulation
current_sim_df = master_df.copy()

# Check if 'Hist_Volatility' is already in master_df (from previous cell execution)
# If it is NOT there, we merge it. If it IS there, we skip merging to avoid duplicates (_x, _y)
if 'Hist_Volatility' not in current_sim_df.columns:
    current_sim_df = pd.merge(current_sim_df, volatility_df, left_on='Team', right_on='Team_Abbr', how='left')

# Fill missing volatility with mean
if 'Hist_Volatility' in current_sim_df.columns:
    mean_vol = current_sim_df['Hist_Volatility'].mean()
    current_sim_df['Hist_Volatility'] = current_sim_df['Hist_Volatility'].fillna(mean_vol)

# Initialize storage
results_wins = {team: [] for team in current_sim_df['Team']}
results_payrolls = {team: [] for team in current_sim_df['Team']}

# Simulation Loop
for i in range(SIMULATIONS):
    # 2. Generate Random Payrolls based on History
    # Mean = Projected 2026 Payroll, Scale = Historical Volatility
    raw_payrolls = np.random.normal(current_sim_df['Payroll'], current_sim_df['Hist_Volatility'])

    # 3. Apply Salary Floor
    final_payrolls = np.maximum(raw_payrolls, SALARY_FLOOR)

    # 4. Calculate Performance (Runs Scored / Runs Allowed)
    # Regression model expects payroll in Millions
    current_runs = ((final_payrolls / 1000000) * slope_rs) + intercept_rs + current_sim_df['Resid_RS']
    current_allowed = ((final_payrolls / 1000000) * slope_ra) + intercept_ra + current_sim_df['Resid_RA']

    # 5. Calculate Win Probability (Pythagorean Expectation)
    win_pcts = (current_runs**1.83) / ((current_runs**1.83) + (current_allowed**1.83))

    # 6. Simulate Season Wins
    season_wins = np.random.binomial(162, win_pcts)

    # 7. Store Results
    for team, wins, pay in zip(current_sim_df['Team'], season_wins, final_payrolls):
        results_wins[team].append(wins)
        results_payrolls[team].append(pay)

# Aggregate Results
HIST_FLOOR_DF = pd.DataFrame({
    'Team': results_wins.keys(),
    'Sim_Wins': [np.mean(val) for val in results_wins.values()],
    'Sim_Avg_Payroll': [np.mean(val) for val in results_payrolls.values()]
})

# Merge with original statistics for comparison
HIST_FLOOR_DF = pd.merge(HIST_FLOOR_DF, current_sim_df[['Team', 'W', 'Payroll']], on='Team')

# Calculate Diffs
HIST_FLOOR_DF['Win_Diff'] = HIST_FLOOR_DF['Sim_Wins'] - HIST_FLOOR_DF['W']
HIST_FLOOR_DF['Payroll_Diff'] = (HIST_FLOOR_DF['Sim_Avg_Payroll'] - HIST_FLOOR_DF['Payroll']) / 1000000

# Sort for display
HIST_FLOOR_DF = HIST_FLOOR_DF.sort_values('Win_Diff', ascending=False)

print("Simulation Complete. Top 5 Gainers:")
display(HIST_FLOOR_DF.head())

print("\n--- Team Historical Volatility (Sorted) ---")
display(current_sim_df[['Team', 'Hist_Volatility']].sort_values('Hist_Volatility', ascending=False).head())
display(current_sim_df[['Team', 'Hist_Volatility']].sort_values('Hist_Volatility', ascending=False).tail())

In [ ]:
### Plot 1: Bar Plot (Win Differential) ###
plt.figure(figsize=(14, 8))
# Color logic: Green for positive win change, Red for negative
colors = ['#d62728' if x < 0 else '#2ca02c' for x in HIST_FLOOR_DF['Win_Diff']]

sns.barplot(
    x='Win_Diff',
    y='Team',
    hue='Team',
    data=HIST_FLOOR_DF,
    palette=colors,
    legend=False
)

plt.title(f"Impact of Salary Floor (${SALARY_FLOOR/1_000_000:.1f}M) w/ Historical Volatility", fontsize=16)
plt.xlabel("Change in Wins", fontsize=12)
plt.axvline(0, color='black', linewidth=1)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()



# --- Summary ---
print("--- Impact Summary ---")
print("1. High Volatility Teams: Teams with historically erratic spending (high std dev) trigger the floor less predictably.")
print("2. Low Volatility Cheap Teams: Teams that consistently spend little (low std dev) are forced to increase payroll significantly every season, leading to consistent win gains.")
print("3. Efficiency: Notice that some teams pay a lot more (Right side of X-axis) but don't gain as many wins (low Y-axis) due to poor existing efficiency (residuals).")

In [ ]:
# --- 1. DEFINE PARK FACTORS (2025 Updates) ---
# 1.00 = Neutral, 1.10 = +10% Runs, 0.90 = -10% Runs
park_factors = {
    'COL': 1.15, 'ATH': 1.08, 'DET': 1.05, 'CIN': 0.99, 'LAD': 1.04,
    'BOS': 1.03, 'TBR': 1.02, 'MIN': 1.01, 'ARI': 1.02, 'KCR': 0.97,
    'CHW': 0.98, 'BAL': 1.03, 'CLE': 0.95, 'ATL': 1.01, 'TOR': 1.03,
    'PIT': 0.96, 'PHI': 1.02, 'HOU': 0.97, 'LAA': 1.01, 'SFG': 0.99,
    'MIL': 0.98, 'MIA': 0.97, 'NYY': 0.99, 'WSN': 1.01, 'SDP': 0.95,
    'STL': 0.97, 'CHC': 0.98, 'NYM': 0.99, 'TEX': 0.91, 'SEA': 0.91
}

# --- 2. APPLY FACTORS TO MASTER DATAFRAME ---
master_df['Park_Factor'] = master_df['Team'].map(park_factors).fillna(1.00)

# --- 3. RE-CALCULATE RESIDUALS ---
# We isolate "Skill" by subtracting the "Park Boost" from the "Actual Runs"

# Step A: Calculate what they SHOULD have scored on a NEUTRAL field (based on money)
# (Using the regression slopes you calculated earlier)
neutral_expected_rs = ((master_df['Payroll'] / 1000000) * slope_rs) + intercept_rs
neutral_expected_ra = ((master_df['Payroll'] / 1000000) * slope_ra) + intercept_ra

# Step B: Adjust that expectation for their specific STADIUM
park_adjusted_exp_rs = neutral_expected_rs * master_df['Park_Factor']
park_adjusted_exp_ra = neutral_expected_ra * master_df['Park_Factor']

# Step C: The Difference is the new "True Skill" Residual
# Residual = Actual - (Money + Stadium)
master_df['Resid_RS'] = master_df['RS'] - park_adjusted_exp_rs
master_df['Resid_RA'] = master_df['RA'] - park_adjusted_exp_ra

# --- 4. VERIFICATION ---

print(f"Columns: {master_df.columns.tolist()}")

# Check the biggest shifts (Colorado should have a much smaller residual now)
print("\n--- NEW RESIDUALS (Top 5 'Overachievers' after Park Adjustment) ---")
display(master_df[['Team', 'RS', 'Park_Factor', 'Resid_RS']].sort_values('Resid_RS', ascending=False).head(5))

print("\n--- NEW RESIDUALS (Top 5 'Underachievers' after Park Adjustment) ---")
display(master_df[['Team', 'RS', 'Park_Factor', 'Resid_RS']].sort_values('Resid_RS', ascending=True).head(5))

In [ ]:
### PARK FACTOR IMPACT SIMULATION (WITH SALARY FLOOR) ###

# --- CONFIGURATION ---
SIMULATIONS = 1000
SALARY_FLOOR = SALARY_FLOOR

print(f"--- Simulating Park Factor Impact with Enforced Salary Floor (${SALARY_FLOOR/1e6:.1f}M) ---")

# 1. PREPARE DATA
park_sim_df = master_df.copy()

# A. Apply Salary Floor to Payroll
# If a team spends less than the floor, boost them up. This changes their "Money Expected" runs.
park_sim_df['Floored_Payroll'] = park_sim_df['Payroll'].apply(lambda x: max(x, SALARY_FLOOR))

# B. Calculate "Money-Based" Performance (Base Runs before Park/Skill)
# We use the regression slopes calculated earlier
money_rs = ((park_sim_df['Floored_Payroll'] * park_sim_df['Park_Factor'] / 1000000) * slope_rs) + intercept_rs
money_ra = ((park_sim_df['Floored_Payroll'] * park_sim_df['Park_Factor'] / 1000000) * slope_ra) + intercept_ra

# 2. DEFINE SCENARIOS

# Scenario A: Real Park Factors (But with Floored Payroll)
# Runs = (Money_Exp * Park_Factor) + Skill_Residual
park_sim_df['Real_Floor_RS'] = (money_rs * park_sim_df['Park_Factor']) + park_sim_df['Resid_RS']
park_sim_df['Real_Floor_RA'] = (money_ra * park_sim_df['Park_Factor']) + park_sim_df['Resid_RA']

# Scenario B: Neutral Park Factors (1.0) (With Floored Payroll)
# Runs = (Money_Exp * 1.0) + Skill_Residual
park_sim_df['Neutral_Floor_RS'] = money_rs + park_sim_df['Resid_RS']
park_sim_df['Neutral_Floor_RA'] = money_ra + park_sim_df['Resid_RA']

# Calculate Win Probabilities for both
park_sim_df['Real_Floor_Win_Pct'] = (park_sim_df['Real_Floor_RS']**1.83) / ((park_sim_df['Real_Floor_RS']**1.83) + (park_sim_df['Real_Floor_RA']**1.83))
park_sim_df['Neutral_Floor_Win_Pct'] = (park_sim_df['Neutral_Floor_RS']**1.83) / ((park_sim_df['Neutral_Floor_RS']**1.83) + (park_sim_df['Neutral_Floor_RA']**1.83))

# 3. MONTE CARLO SIMULATION
results_real = {team: [] for team in park_sim_df['Team']}
results_neutral = {team: [] for team in park_sim_df['Team']}

print(f"Simulating {SIMULATIONS} seasons for both scenarios...")

for i in range(SIMULATIONS):
    for idx, row in park_sim_df.iterrows():
        # Scenario A: Real Park + Floor
        wins_real = np.random.binomial(n=162, p=row['Real_Floor_Win_Pct'])
        results_real[row['Team']].append(wins_real)

        # Scenario B: Neutral Park + Floor
        wins_neutral = np.random.binomial(n=162, p=row['Neutral_Floor_Win_Pct'])
        results_neutral[row['Team']].append(wins_neutral)

# 4. AGGREGATE RESULTS
PARK_IMPACT_DF = pd.DataFrame({
    'Team': results_real.keys(),
    'Real_Floor_Wins': [np.mean(val) for val in results_real.values()],
    'Neutral_Floor_Wins': [np.mean(val) for val in results_neutral.values()]
})

PARK_IMPACT_DF = pd.merge(PARK_IMPACT_DF, park_sim_df[['Team', 'Park_Factor', 'Payroll', 'Floored_Payroll']], on='Team')

# 5. CALCULATE IMPACT
# Impact = Wins(With Park) - Wins(Without Park)
PARK_IMPACT_DF['Park_Impact_Wins'] = PARK_IMPACT_DF['Real_Floor_Wins'] - PARK_IMPACT_DF['Neutral_Floor_Wins']
PARK_IMPACT_DF = PARK_IMPACT_DF.sort_values('Park_Impact_Wins', ascending=False)

print("Top 5 Teams Benefiting from their Park (Under Salary Floor):")
display(PARK_IMPACT_DF[["Team","Real_Floor_Wins", "Neutral_Floor_Wins", "Park_Impact_Wins"]].head())

print("Top 5 Teams Hurt by their Park (Under Salary Floor):")
display(PARK_IMPACT_DF[["Team","Real_Floor_Wins", "Neutral_Floor_Wins", "Park_Impact_Wins"]].tail().sort_values('Park_Impact_Wins'))

In [ ]:
# 5. VISUALIZATION
plt.figure(figsize=(14, 8))
colors = ['#d62728' if x < 0 else '#2ca02c' for x in PARK_IMPACT_DF['Park_Impact_Wins']]

# Label with Park Factor for context
plot_data = PARK_IMPACT_DF.copy()
plot_data['Label'] = plot_data['Team'] + " (PF: " + plot_data['Park_Factor'].astype(str) + ")"

sns.barplot(
    x='Park_Impact_Wins',
    y='Label',
    hue='Team',
    data=plot_data,
    palette=colors,
    legend=False
)

plt.title(f"Impact of Stadium Factors with ${SALARY_FLOOR/1000000:.1f}M Salary Floor: Real vs. Neutral Field", fontsize=16)
plt.xlabel("Wins Gained/Lost due to Stadium (Payroll adjusted to Floor)", fontsize=12)
plt.ylabel("Team (Park Factor)", fontsize=12)
plt.axvline(0, color='black', linewidth=1)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
#schedule creation


schedule_df = pd.read_csv("/content/mlb_schedule_2025.csv")

team_map = {
    'Angels': 'LAA', 'Arizona Diamondbacks': 'ARI', 'Astros': 'HOU', 'Athletics': 'ATH', 'Atlanta Braves': 'ATL',
    'Baltimore Orioles': 'BAL', 'Blue Jays': 'TOR', 'Boston Red Sox': 'BOS', 'Braves': 'ATL', 'Brewers': 'MIL',
    'Cardinals': 'STL', 'Chicago Cubs': 'CHC', 'Chicago White Sox': 'CHW', 'Cincinnati Reds': 'CIN', 'Cleveland Guardians': 'CLE',
    'Cleveland Indians': 'CLE', 'Colorado Rockies': 'COL', 'Cubs': 'CHC', 'Detroit Tigers': 'DET', 'Diamondbacks': 'ARI',
    'Dodgers': 'LAD', 'Giants': 'SFG', 'Guardians': 'CLE', 'Houston Astros': 'HOU', 'Kansas City Royals': 'KCR',
    'Los Angeles Angels': 'LAA', 'Los Angeles Dodgers': 'LAD', 'Mariners': 'SEA', 'Marlins': 'MIA', 'Mets': 'NYM',
    'Miami Marlins': 'MIA', 'Milwaukee Brewers': 'MIL', 'Minnesota Twins': 'MIN', 'Nationals': 'WSN', 'New York Mets': 'NYM',
    'New York Yankees': 'NYY', 'Oakland Athletics': 'ATH', 'Orioles': 'BAL', 'Padres': 'SDP', 'Philadelphia Phillies': 'PHI',
    'Phillies': 'PHI', 'Pirates': 'PIT', 'Pittsburgh Pirates': 'PIT', 'Rangers': 'TEX', 'Rays': 'TBR',
    'Red Sox': 'BOS', 'Reds': 'CIN', 'Rockies': 'COL', 'Royals': 'KCR', 'San Diego Padres': 'SDP',
    'San Francisco Giants': 'SFG', 'Seattle Mariners': 'SEA', 'St. Louis Cardinals': 'STL', 'Tampa Bay Rays': 'TBR', 'Texas Rangers': 'TEX',
    'Tigers': 'DET', 'Toronto Blue Jays': 'TOR', 'Twins': 'MIN', 'Washington Nationals': 'WSN', 'White Sox': 'CHW',
    'Yankees': 'NYY', 'D-backs' : 'ARI'
}

schedule_df["Home"] = schedule_df["Home"].map(lambda x: team_map.get(x, x))
schedule_df["Away"] = schedule_df["Away"].map(lambda x: team_map.get(x, x))



In [ ]:
SIMULATIONS = 5000

SALARY_FLOOR = SALARY_FLOOR

print(f"--- STARTING FINAL GAME-LEVEL SIMULATION ---")
print(f"Constraints: Floor=${SALARY_FLOOR/1e6:.1f}M | Cap=${SALARY_CAP/1e6:.1f}M")
print("Factors: Historical Volatility, Dynasty Efficiency, Park Factors, Redistribution")


schedule = list(zip(schedule_df["Home"], schedule_df["Away"]))

teams = master_df['Team'].values


HOME_ADV = 0.04  # ~54% MLB home advantage

team_to_idx = {team: i for i, team in enumerate(teams)}

def win_probability(home_idx, away_idx, strengths):
    s_home = strengths[home_idx] + HOME_ADV
    s_away = strengths[away_idx]
    return s_home / (s_home + s_away)

def simulate_season_games(schedule, strengths):
    wins = np.zeros(len(strengths))

    for home, away in schedule:
        h = team_to_idx[home]
        a = team_to_idx[away]

        p_home = win_probability(h, a, strengths)

        if random.random() < p_home:
            wins[h] += 1
        else:
            wins[a] += 1

    return wins

results_final = {team: [] for team in teams}
results_payrolls = {team: [] for team in teams}

for sim in range(SIMULATIONS):

    # 1. Volatility: simulate payrolls
    sim_payrolls = np.random.normal(
        master_df['Payroll'],
        master_df['Hist_Volatility']
    )

    # 2. Salary Floor
    sim_payrolls = np.maximum(sim_payrolls, SALARY_FLOOR)

    # 3. Efficiency Adjustment
    eff_payrolls = sim_payrolls * master_df['Efficiency_Factor']

    # 4. Convert Payroll → Runs
    money_rs = ((eff_payrolls / 1_000_000) * slope_rs) + intercept_rs
    money_ra = ((eff_payrolls / 1_000_000) * slope_ra) + intercept_ra

    # 5. Park Factors
    park_rs = money_rs * master_df['Park_Factor']
    park_ra = money_ra * master_df['Park_Factor']

    # 6. Add residual skill
    sim_rs = park_rs + master_df['Resid_RS']
    sim_ra = park_ra + master_df['Resid_RA']

    # 7. Pythagorean expectation → latent strength
    win_pcts = (sim_rs**1.83) / ((sim_rs**1.83) + (sim_ra**1.83))

    strengths = win_pcts.values

    # Normalize strengths (important stability step)
    strengths = strengths / np.mean(strengths)

    # GAME-LEVEL SEASON SIMULATION
    season_wins = simulate_season_games(schedule, strengths)

    # Store results
    for idx, team in enumerate(teams):
        results_final[team].append(season_wins[idx])
        results_payrolls[team].append(sim_payrolls[idx])

FINAL_SIM_DF = pd.DataFrame({
    'Team': results_final.keys(),
    'Sim_Wins': [np.mean(val) for val in results_final.values()],
    'Sim_Payroll': [np.mean(val) for val in results_payrolls.values()]
})

FINAL_SIM_DF = pd.merge(
    FINAL_SIM_DF,
    master_df[['Team','W','Payroll','Efficiency_Factor','Park_Factor']],
    on='Team'
)

FINAL_SIM_DF['Win_Diff'] = FINAL_SIM_DF['Sim_Wins'] - FINAL_SIM_DF['W']
FINAL_SIM_DF['Payroll_Cut'] = FINAL_SIM_DF['Payroll'] - FINAL_SIM_DF['Sim_Payroll']

FINAL_SIM_DF = FINAL_SIM_DF.sort_values('Win_Diff', ascending=False)

print("\n--- FINAL RESULTS (Top 5 Gainers) ---")
display(FINAL_SIM_DF[['Team','Sim_Wins','W','Win_Diff','Payroll']].head())

print("\n--- FINAL RESULTS (Top 5 Losers) ---")
display(
    FINAL_SIM_DF[['Team','Sim_Wins','W','Win_Diff','Payroll']]
    .tail()
    .sort_values('Win_Diff')
)


In [ ]:
# --- VISUALIZATION ---
plt.figure(figsize=(14, 8))
colors = ['#d62728' if x < 0 else '#2ca02c' for x in FINAL_SIM_DF['Win_Diff']]

sns.barplot(
    x='Win_Diff',
    y='Team',
    hue='Team',
    data=FINAL_SIM_DF,
    palette=colors,
    legend=False
)

plt.title("Impact of All Factors (2027 Projection)", fontsize=16)
plt.xlabel("Projected Change in Wins", fontsize=12)
plt.axvline(0, color='black', linewidth=1)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

# 1. Convert Raw Results to DataFrame (Rows=Simulations, Cols=Teams)
raw_sim_df = pd.DataFrame(results_final)

# 2. Calculate Key Metrics per Team
# We use describe() to get count, mean, std, min, percentiles, max
team_metrics = raw_sim_df.describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]).T

# Sort by Mean Wins for better visualization
team_metrics = team_metrics.sort_values('mean', ascending=False)

print("--- Key Simulation Metrics by Team (Sorted by Projected Wins) ---")
# Displaying Mean, Std Dev, and the 90% confidence interval (5% to 95%)
display(team_metrics[['mean', 'std', '5%', '50%', '95%', 'min', 'max']].head(10))


# 3. Visualization: Box Plot (Distribution of Outcomes)
plt.figure(figsize=(16, 8))

# Define colors: Green if mean > 81, Red otherwise


sns.boxplot(data=raw_sim_df, order=team_metrics.index, color="lightskyblue")
plt.title(f"Distribution of Simulated Season Outcomes ({SIMULATIONS} Iterations)\nBox = 25th-75th %ile | Whiskers = Range", fontsize=16)
plt.ylabel("Simulated Wins", fontsize=12)
plt.xlabel("Team", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()

# 4. League Parity Analysis
# Calculate the standard deviation of wins *across the league* for each individual season simulation
# Lower Std Dev = More Parity (Teams are bunched closer together)
season_std_devs = raw_sim_df.std(axis=1)

plt.figure(figsize=(10, 6))
sns.histplot(season_std_devs, bins=30, kde=True, color='lightskyblue')
plt.title("Projected League Parity: Distribution of Standings Spread", fontsize=16)
plt.xlabel("Standard Deviation of Wins in a Season (Lower = More Parity)", fontsize=12)
plt.ylabel("Frequency (Count of Simulations)", fontsize=12)
plt.axvline(season_std_devs.mean(), color='red', linestyle='--', label=f'Avg Std Dev: {season_std_devs.mean():.2f}')
plt.legend()
plt.show()

In [ ]:
# --- CONSOLIDATE ALL SIMULATION RESULTS INTO MASTER_DF ---

# 1. Define the simulation result dataframes and their suffixes
# (Checking if they exist to avoid errors if a step was skipped)
sim_results = []
if 'SALARY_CAP_DF' in locals(): sim_results.append((SALARY_CAP_DF, 'Cap'))
if 'SALARY_FLOOR_DF' in locals(): sim_results.append((SALARY_FLOOR_DF, 'Floor'))
if 'HABITS_FLOOR_DF' in locals(): sim_results.append((HABITS_FLOOR_DF, 'Habits'))
if 'HIST_FLOOR_DF' in locals(): sim_results.append((HIST_FLOOR_DF, 'Hist_Vol'))
if 'DESIRE_DF' in locals(): sim_results.append((DESIRE_DF, 'Dynasty'))
if 'FINAL_SIM_DF' in locals(): sim_results.append((FINAL_SIM_DF, 'Final'))

# 2. Reset master_df to base columns to ensure a clean start (if re-running)
# We look for the core columns that should exist from previous steps
base_cols = [
    'Team', 'W', 'L', 'RS', 'RA', 'Payroll', 'Win_Pct',
    'Hist_Volatility', 'Efficiency_Factor', 'Park_Factor',
    'Resid_RS', 'Resid_RA'
]

# Ensure we are using the columns available in the current master_df
available_cols = [c for c in base_cols if c in master_df.columns]
master_df = master_df[available_cols].copy()

# 3. Merge Loop
for df, suffix in sim_results:
    if 'Win_Diff' in df.columns:
        # Select only Team and Win_Diff, rename Win_Diff to be specific
        temp_df = df[['Team', 'Win_Diff']].rename(columns={'Win_Diff': f'Diff_{suffix}'})
        # Merge
        master_df = pd.merge(master_df, temp_df, on='Team', how='left')

# 4. Display Final Consolidated Data
print("--- CONSOLIDATED MASTER DATASET (Inputs & Simulation Results) ---")
display(master_df.head())


In [ ]:
DIVISIONS = {
    "AL East": ["NYY","BOS","TOR","BAL","TBR"],
    "AL Central": ["CLE","DET","KCR","MIN","CHW"],
    "AL West": ["HOU","SEA","TEX","LAA","ATH"],
    "NL East": ["ATL","PHI","NYM","MIA","WSN"],
    "NL Central": ["MIL","CHC","CIN","PIT","STL"],
    "NL West": ["LAD","ARI","SDP","SFG","COL"]
}

division_tables = {}

wins_df = pd.DataFrame(results_final)

for division, teams in DIVISIONS.items():

    div_wins = wins_df[teams]

    # Rank teams within division each simulation
    div_ranks = div_wins.rank(axis=1, ascending=False, method='average')

    division_projection = pd.DataFrame({
        "Team": teams,
        "Proj_Wins": div_wins.mean().values,
        "Win_SD": div_wins.std().values,
        "Avg_Div_Finish": div_ranks.mean().values,
        "Win_Division_%": (div_ranks == 1).mean().values * 100,
        "Finish_Last_%": (div_ranks == len(teams)).mean().values * 100
    })

    division_projection = division_projection.sort_values(
        "Proj_Wins", ascending=False
    ).reset_index(drop=True)

    division_tables[division] = division_projection

print("=== PROJECTED DIVISION STANDINGS ===\n")

for division, table in division_tables.items():
    print(f"\n--- {division} ---")
    display(table.round(2))

